# Fase 1: limpieza de datos

Este notebook sirve para **explorar y probar** la limpieza con los datos simulados
(`data/simulated/`). Cuando una etapa funcione, su código se pasa a
`src/occupancy/cleaning.py`.

Orden de las etapas (ver el documento del pipeline):

0. Cargar los datos y convertir las fechas
1. Duplicados
2. Control de rango
3. Picos aislados
4. Huecos
5. Suavizado (media móvil)

Las celdas marcadas como **Comprobación** ya están escritas: ejecútalas después de
tu código para saber si el resultado es correcto.

In [ ]:
import json
from pathlib import Path

import pandas as pd

# Buscar la raíz del repositorio (la carpeta que contiene config.yaml)
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "simulated" / "readings.json"
FAULTS_PATH = ROOT / "data" / "simulated" / "faults.json"
print("Datos:", DATA_PATH)

## 0. Cargar los datos

El archivo tiene el mismo formato que la respuesta de la API. Esta celda ya lo lee y
deja en `data` un diccionario con una lista de lecturas por sensor.

In [ ]:
with open(DATA_PATH, encoding="utf-8") as f:
    response = json.load(f)

data = response["data"]
print("Sensores:", list(data))
print("Primera lectura de temp:", data["temp"][0])
print("Primera lectura de PIR:", data["PIR"][0])

### Tu tarea: convertir cada sensor en una tabla

Guarda el resultado en un diccionario llamado **`tables`**: una tabla por sensor
(menos `PIR` y `busy_level`), con las columnas `timestamp` y `value`.

- Cada lista de lecturas se convierte en tabla con `pd.DataFrame`.
- Convierte `timestamp` a fecha con `pd.to_datetime`, indicando el formato
  `"%Y-%m-%d %H:%M:%S.%f"`.

In [ ]:
# Tu código aquí
tables = {}

### Tu tarea: convertir los PIR en una tabla

Guarda el resultado en una tabla llamada **`pir`**, con una fila por marca de tiempo y
las columnas `timestamp`, `pir-1`, `pir-2` y `pir-3`. Convierte también `timestamp`
a fecha.

In [ ]:
# Tu código aquí
pir = None

In [ ]:
# Comprobación de la carga (no hace falta modificarla)
expected = {s: 1067 for s in ["pressure", "hum", "temp", "voc", "airresistance", "mic_vol"]}
expected["co2equivalent"] = 1046

for sensor, n in expected.items():
    table = tables[sensor]
    ok = len(table) == n and pd.api.types.is_datetime64_any_dtype(table["timestamp"])
    print(f"{sensor:14} filas={len(table):5}  esperado={n:5}  {'OK' if ok else 'REVISAR'}")

ok_pir = (len(pir) == 1067
          and set(pir.columns) == {"timestamp", "pir-1", "pir-2", "pir-3"}
          and pd.api.types.is_datetime64_any_dtype(pir["timestamp"]))
print(f"{'PIR':14} filas={len(pir):5}  esperado= 1067  {'OK' if ok_pir else 'REVISAR'}")

## 1.1 Duplicados

Un duplicado de QoS 1 es una fila **exactamente igual** a otra: misma hora y mismos
valores. Quítalos con `drop_duplicates` sobre la fila completa, no solo sobre la hora.

Guarda el resultado en las mismas variables (`tables` y `pir`).

In [ ]:
# Tu código aquí

In [ ]:
# Comprobación de duplicados (no hace falta modificarla)
expected = {s: 1059 for s in ["pressure", "hum", "temp", "voc", "airresistance", "mic_vol"]}
expected["co2equivalent"] = 1038

for sensor, n in expected.items():
    table = tables[sensor]
    ok = len(table) == n and table.duplicated().sum() == 0
    print(f"{sensor:14} filas={len(table):5}  esperado={n:5}  {'OK' if ok else 'REVISAR'}")

ok_pir = len(pir) == 1059 and pir.duplicated().sum() == 0
print(f"{'PIR':14} filas={len(pir):5}  esperado= 1059  {'OK' if ok_pir else 'REVISAR'}")

## 1.2 Control de rango

Convierte en vacío (`NaN`) los valores imposibles para cada sensor, **sin borrar la
fila**. Los límites reales saldrán de la hoja de datos de cada sensor; para los datos
simulados basta con detectar los valores claramente imposibles.

Herramienta: `mask`.

In [ ]:
# Tu código aquí

## 1.3 Picos aislados

Convierte en vacío los valores que se disparan un instante y vuelven, comparando cada
valor con sus vecinos. Sobre todo en `mic_vol`.

Herramientas: `rolling` y `mask`.

In [ ]:
# Tu código aquí

## 1.4 Huecos

Primero mide cada cuánto llegan las lecturas (`diff` sobre `timestamp`). Después
rellena solo los huecos cortos y deja vacíos los largos.

Herramienta: `interpolate` con `method="time"` y un límite.

In [ ]:
# Tu código aquí

## 1.5 Suavizado

Media móvil que **solo mira hacia atrás**, aplicada a `mic_vol`, `co2equivalent` y `voc`.

Herramienta: `rolling` con una ventana de tiempo.

In [ ]:
# Tu código aquí

## Inspección visual

Dibuja cada sensor antes y después de la limpieza para comprobar que no queda nada raro.
Es la parte de "inspección humana" de la diapositiva 25.

In [ ]:
# Tu código aquí